# Setup

Runs on **Google Colab** or **locally** (`COLAB_RELEASE_TAG` is used to detect Colab). Paths, Drive mount, clone, and dataset unzip run only when needed.

In [16]:
import os
from pathlib import Path
import yaml
import subprocess
import sys
from datetime import datetime
import zipfile
import copy

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
REPO_URL = "https://github.com/chendwend/thesis-assyrian-relief.git"
TIMESTAMP = datetime.now().strftime("%d-%m_%H-%M-%S")


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "configs" / "style_dinov2.yaml").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find repo root (missing configs/style_dinov2.yaml). "
        "Cd to the thesis-assyrian-relief clone or open the notebook from that repo."
    )


def sync_notebook_cwd(path: Path) -> None:
    """Match Python and IPython shell cwd so `!command` cells run in the repo."""
    os.chdir(path)
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("cd", str(path))
    except ImportError:
        pass


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_DIR = Path("/content/thesis-assyrian-relief")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Graduate_Studies/Thesis")
    IMAGE_ROOT = Path("/content/dataset")
    
    OUTPUT_ROOT = DRIVE_ROOT / "outputs" / TIMESTAMP
else:
    REPO_DIR = find_repo_root()
    sync_notebook_cwd(REPO_DIR)
    import yaml

    with open(REPO_DIR / "configs" / "style_dinov2.yaml", encoding="utf-8") as f:
        _model_cfg = yaml.safe_load(f)

    with open(REPO_DIR / "configs" / "paths.yaml", encoding="utf-8") as f:
        _paths_cfg = yaml.safe_load(f)
        
    IMAGE_ROOT = Path(_paths_cfg["data"]["local"]["dataset_root"]).expanduser()
    OUTPUT_ROOT = (REPO_DIR / "outputs"/TIMESTAMP).resolve()



# Unzip dataset if in Colab
if IN_COLAB:
    for zip_path in (Path("/content/dataset.zip"), Path.cwd() / "dataset.zip"):
        if zip_path.is_file():
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(Path("/content"))
            zip_path.unlink(missing_ok=True)
            print(f"Extracted and removed {zip_path}")
            break
    else:
        print("No dataset.zip found; skip unzip (use Drive-mounted data or upload zip).")
else:
    print("Local: skip dataset unzip.")


# Clone repo if in Colab
if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], cwd="/content", check=True)
    sync_notebook_cwd(REPO_DIR)
else:
    if not (REPO_DIR / "configs" / "style_dinov2.yaml").is_file():
        raise FileNotFoundError(REPO_DIR / "configs" / "style_dinov2.yaml")

print("CWD:", Path.cwd())

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"IMAGE_ROOT={IMAGE_ROOT}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")

/home/kostya/projects/thesis-assyrian-relief
Local: skip dataset unzip.
CWD: /home/kostya/projects/thesis-assyrian-relief
IN_COLAB=False
REPO_DIR=/home/kostya/projects/thesis-assyrian-relief
IMAGE_ROOT=/mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
OUTPUT_ROOT=/home/kostya/projects/thesis-assyrian-relief/outputs/14-05_18-47-40


In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync"], cwd=str(REPO_DIR), check=True)

In [ ]:
base_cfg_path = Path("configs/style_dinov2.yaml")
runtime_cfg_path = Path(f"{OUTPUT_ROOT}/style_dinov2_runtime.yaml")
val_eval_cfg_path = Path(f"{OUTPUT_ROOT}/style_dinov2_eval_val.yaml")
test_eval_cfg_path = Path(f"{OUTPUT_ROOT}/style_dinov2_eval_test.yaml")

with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["image_root"] = str(IMAGE_ROOT)

cfg["outputs"] = {}
cfg["outputs"]["checkpoint_path"] = str(OUTPUT_ROOT / "checkpoints" / "dinov2_probe.pt")
cfg["outputs"]["history_path"] = str(OUTPUT_ROOT / "checkpoints" / "dinov2_probe.history.csv")
cfg["outputs"]["training_curve_path"] = str(OUTPUT_ROOT / "plots" / "train_curves.png")

# Default evaluation choice. You can later change this after inspecting val metrics.
cfg["evaluation"] = {}
cfg["evaluation"]["relief_aggregation"] = "mean_logits"

# Other outputs used by separate scripts can stay here
cfg["outputs"]["umap_html_path"] = str(OUTPUT_ROOT / "umap" / "test_umap.html")
cfg["outputs"]["umap_csv_path"] = str(OUTPUT_ROOT / "umap" / "test_umap.csv")
cfg["outputs"]["retrieval_metrics_path"] = str(OUTPUT_ROOT / "retrieval" / "test_metrics.json")
cfg["outputs"]["retrieval_top1_path"] = str(OUTPUT_ROOT / "retrieval" / "test_top1.csv")
cfg["outputs"]["retrieval_topk_path"] = str(OUTPUT_ROOT / "retrieval" / "test_topk.csv")
cfg["outputs"]["retrieval_failures_path"] = str(OUTPUT_ROOT / "retrieval" / "test_failures.csv")


def make_eval_cfg(base_cfg: dict, split: str) -> dict:
    eval_cfg = copy.deepcopy(base_cfg)

    eval_cfg["outputs"]["confusion_matrix_path"] = str(
        OUTPUT_ROOT / "plots" / f"confusion_matrix_{split}.png"
    )
    eval_cfg["outputs"]["eval_metrics_path"] = str(
        OUTPUT_ROOT / "eval" / f"{split}_metrics.json"
    )
    eval_cfg["outputs"]["eval_retrieval_path"] = str(
        OUTPUT_ROOT / "eval" / f"{split}_retrieval.csv"
    )
    eval_cfg["outputs"]["relief_predictions_path"] = str(
        OUTPUT_ROOT / "eval" / f"{split}_relief_predictions.csv"
    )

    return eval_cfg


train_cfg = cfg
val_eval_cfg = make_eval_cfg(cfg, "val")
test_eval_cfg = make_eval_cfg(cfg, "test")


# create dirs
for current_cfg in [train_cfg, val_eval_cfg, test_eval_cfg]:
    for _, out_path in current_cfg["outputs"].items():
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)

# dump configs
with open(runtime_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)

with open(val_eval_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(val_eval_cfg, f, sort_keys=False)

with open(test_eval_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(test_eval_cfg, f, sort_keys=False)

print(f"Saved train runtime config to: {runtime_cfg_path}")
print(f"Saved val eval config to: {val_eval_cfg_path}")
print(f"Saved test eval config to: {test_eval_cfg_path}")

Saved runtime config to: /home/kostya/projects/thesis-assyrian-relief/outputs/14-05_13-26-45/style_dinov2_runtime.yaml

-------------------------------------------------- CONFIG --------------------------------------------------
data:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: '-'
splits:
  train: train
  val: val
  test: test
model:
  model_name: dinov2_vits14
  emb_dim: 256
train:
  batch_size: 16
  num_workers: 0
  num_epochs: 15
  lr: 0.001
  weight_decay: 0.0001
outputs:
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/14-05_13-26-45/checkpoints/dinov2_probe.pt
  history_path: /home/kostya/projects/thesis-assyrian-relief/outputs/14-05_13-26-45/checkpoints/dinov2_probe.history.csv
  training_curve_path: /home/kostya/projects/thesis-assyrian-relief/outputs/14-05_13-26-45/plots/train_curves.png
  confusion_matrix_path: /home/kostya/projects/thesis-assyrian-relief

# Training

In [ ]:
!uv run python scripts/train_style.py \
  --config {runtime_cfg_path}

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  val_split: val
  model_name: dinov2_vits14
  emb_dim: 256
  batch_size: 16
  num_workers: 0
  num_epochs: 15
  lr: 0.001
  weight_decay: 0.0001
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/checkpoints/dinov2_probe_v2.pt
  history_path: /home/kostya/projects/thesis-assyrian-relief/outputs/checkpoints/dinov2_probe_v2.history.csv
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2, 'Sennacherib': 3, 'Tiglath-Pileser III': 4}
Building train loader...
Using device: cuda
Using cache found in /home/kostya/.cache/torch/hub/facebookresearch_dinov2_main
class_weights: tensor([0.5489, 0.5838, 1.2169, 2.4634, 4.2083], device='cuda:0')
Trainable parameter tensors: 6
Training model...
Epoch 01 | lr=1.00e-03 | train_loss=1.4879 train_acc=0.5188 train_f1=0.4594 | val_loss=1

# Validation

In [ ]:
!uv run python scripts/eval_style.py \
  --config {val_eval_cfg_path} \
  --eval-split val

# Evaluation

In [ ]:
!uv run python scripts/eval_style.py \
  --config {test_eval_cfg_path} \
  --eval-split test

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  eval_split: test
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/checkpoints/dinov2_probe_v2.pt
  batch_size: 16
  num_workers: 0
  metrics_out: /home/kostya/projects/thesis-assyrian-relief/outputs/eval/test_metrics_v2.json
  retrieval_out: /home/kostya/projects/thesis-assyrian-relief/outputs/eval/test_retrieval_v2.csv
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2, 'Sennacherib': 3, 'Tiglath-Pileser III': 4}
model_name: dinov2_vits14
emb_dim: 256
Using device: cuda
Using cache found in /home/kostya/.cache/torch/hub/facebookresearch_dinov2_main
                                                                                
=== Evaluation Summary ===
{
  "image_level": {
    "accuracy": 0.6929824561403509,
    "macro_f1": 0.6268846600553918,
    "loss": 1.

# UMAP

In [20]:
!uv run python scripts/umap_style.py \
  --config configs/style_dinov2_runtime.yaml \
  --umap-fit-split train \
  --umap-plot-splits train val test \
  --html-out outputs/umap/umap_fit-train_plot-val-test_v2.html

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  eval_split: test
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/checkpoints/dinov2_probe_v2.pt
  batch_size: 16
  num_workers: 0
  html_out: outputs/umap/umap_fit-train_plot-val-test_v2.html
  csv_out: /home/kostya/projects/thesis-assyrian-relief/outputs/umap/test_umap_v2.csv
  highlight_relief_ids: []
  umap_fit_split: train
  umap_plot_splits: ['train', 'val', 'test']
Using device: cuda
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2, 'Sennacherib': 3, 'Tiglath-Pileser III': 4}
Using cache found in /home/kostya/.cache/torch/hub/facebookresearch_dinov2_main
Extracting embeddings for split='test'...
Extracting embeddings for split='train'...
Extracting embeddings for split='val'...
/home/kostya/projects/thesis-assyrian-relief/.venv/lib/python3.13/site-pack

# Retrieval Analysis

In [21]:
!uv run python scripts/retrieval_analysis.py \
  --config configs/style_dinov2_runtime.yaml \
  --eval-split test

Resolved config:
  csv_path: data/splits/image_level_dataset_v2.csv
  image_root: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2
  filename_sep: -
  train_split: train
  eval_split: test
  checkpoint_path: /home/kostya/projects/thesis-assyrian-relief/outputs/checkpoints/dinov2_probe_v2.pt
  batch_size: 16
  num_workers: 0
  topk: 5
  metrics_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_metrics_v2.json
  top1_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_top1_v2.csv
  topk_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_topk_v2.csv
  failures_out: /home/kostya/projects/thesis-assyrian-relief/outputs/retrieval/test_failures_v2.csv
Using device: cuda
class_to_idx: {'Ashurbanipal': 0, 'Ashurnasirpal II': 1, 'Sargon II': 2, 'Sennacherib': 3, 'Tiglath-Pileser III': 4}
Using cache found in /home/kostya/.cache/torch/hub/facebookresearch_dinov2_main
Extracting train embeddings...
Extracting eval embed

## test

In [ ]:
!uv run python scripts/train_style.py \
  --config configs/style_dinov2_runtime.yaml \
  --num-epochs 1 \
  --batch-size 8 \
  --num-workers 2